# 🔢 MNIST 手写数字识别 — CNN 多分类

**目标**：基于 MNIST 数据集，使用 TensorFlow/Keras 构建 CNN 卷积神经网络，实现对手写数字 0-9 的识别分类。

**数据集**：
| 文件 | 样本数 | 数据形状 | 类别 |
|------|--------|----------|------|
| `mnist.pkl.gz` | 70,000 | 28×28 灰度图 | 0-9 共 10 类 |

> ⚠️ **关于数据格式**：该数据集已预归一化到 [0, 1] 范围，**无需再做 /255 操作**。

## 目录

1. 环境准备
2. 数据加载与探索
3. 数据预处理
4. 数据可视化
5. 模型搭建（逐层详解）
6. 模型训练
7. 模型评估（混淆矩阵 + 分类报告）
8. 预测验证与错误分析
9. 保存与加载模型
10. 调参实验区

---
## 1. 环境准备

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle, gzip, os

# 中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow {tf.__version__}  |  NumPy {np.__version__}")

---
## 2. 数据加载与探索

In [ ]:
data_path = os.path.join('..', 'mnist', 'mnist.pkl.gz')
with gzip.open(data_path, 'rb') as f:
    training_data, validation_data, test_data = pickle.load(f, encoding='latin1')

X_train_raw, y_train_raw = training_data
X_val_raw,   y_val_raw   = validation_data
X_test_raw,  y_test_raw  = test_data

print(f"训练集: {len(X_train_raw):,}  |  验证集: {len(X_val_raw):,}  |  测试集: {len(X_test_raw):,}")
print(f"原始形状: {X_train_raw.shape}（扁平向量 784=28×28）")
print(f"值域: [{X_train_raw.min():.4f}, {X_train_raw.max():.4f}]  ← 已预归一化到 [0,1]")

In [ ]:
# 查看标签分布
labels, counts = np.unique(y_train_raw, return_counts=True)
for l, c in zip(labels, counts):
    bar = '█' * (c // 300)
    print(f"  数字 {l}: {c:5,}  {bar}")
print(f"\n类别均衡性: 最少 {counts.min()}, 最多 {counts.max()}, 比值 {counts.max()/counts.min():.2f}")

---
## 3. 数据预处理

| 步骤 | 操作 | 说明 |
|------|------|------|
| reshape | (N, 784) → (N, 28, 28, 1) | 还原为 2D 图像 + 通道维度 |
| 类型转换 | int → float32 | Keras 需要 float 输入 |
| 合并重分 | 训练+验证 → 9:1 重分 | 统一划分，保证随机性 |

> ⚠️ **不需要除以 255**：数据已在 [0,1] 范围。

In [ ]:
# Reshape + 类型转换（不除以 255！数据已在 [0,1] 范围内）
X_train_raw = X_train_raw.reshape(-1, 28, 28, 1).astype('float32')
X_val_raw   = X_val_raw.reshape(-1, 28, 28, 1).astype('float32')
X_test_raw  = X_test_raw.reshape(-1, 28, 28, 1).astype('float32')

y_train_raw = np.array(y_train_raw)
y_val_raw   = np.array(y_val_raw)
y_test_raw  = np.array(y_test_raw)

# 合并训练+验证集，重新 9:1 划分
X_all = np.concatenate([X_train_raw, X_val_raw], axis=0)
y_all = np.concatenate([y_train_raw, y_val_raw], axis=0)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_all, y_all, test_size=0.1, random_state=42, stratify=y_all
)
X_test, y_test = X_test_raw, y_test_raw

print(f"训练集: {X_train.shape}  |  验证集: {X_valid.shape}  |  测试集: {X_test.shape}")
print(f"值域: [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"类别数: {len(np.unique(y_train))}  (0-9)")

---
## 4. 数据可视化

随机展示 25 张训练样本，直观感受数据。

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(8, 8))
indices = np.random.choice(len(X_train), 25, replace=False)

for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[indices[i]].reshape(28, 28), cmap='gray')
    ax.set_title(f'标签: {y_train[indices[i]]}', fontsize=12)
    ax.axis('off')

fig.suptitle('MNIST 训练集样本（随机 25 张）', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. 模型搭建

采用经典的 CNN 架构：两层卷积提取特征 → 展平 → 全连接分类 → Softmax 输出 10 类概率。

| 层 | 类型 | 参数 | 输出形状 | 参数量 |
|----|------|------|----------|--------|
| 1 | Conv2D + ReLU | 32核, 3×3 | 26×26×32 | 320 |
| 2 | MaxPooling2D | 2×2 | 13×13×32 | 0 |
| 3 | Conv2D + ReLU | 64核, 3×3 | 11×11×64 | 18,496 |
| 4 | MaxPooling2D | 2×2 | 5×5×64 | 0 |
| 5 | Flatten | - | 1600 | 0 |
| 6 | Dense + ReLU | 128 | 128 | 204,928 |
| 7 | Dropout | 0.5 | 128 | 0 |
| 8 | Dense + Softmax | 10 | 10 | 1,290 |
| **合计** | | | | **~225,034** |

> **为什么用 CNN 而不是全连接？** 卷积保留图像的 2D 空间结构，参数量远小于全连接（784→128 就需要 100k+ 参数），且能捕捉局部特征（边缘、纹理等）。

In [ ]:
model = keras.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1), name='conv1'),
    layers.MaxPooling2D((2,2), name='pool1'),
    layers.Conv2D(64, (3,3), activation='relu', name='conv2'),
    layers.MaxPooling2D((2,2), name='pool2'),
    layers.Flatten(name='flatten'),
    layers.Dense(128, activation='relu', name='fc1'),
    layers.Dropout(0.5, name='dropout'),
    layers.Dense(10, activation='softmax', name='output')
], name='MNIST_CNN')

model.summary()

### 5.1 编译模型

| 配置项 | 选择 | 原因 |
|--------|------|------|
| 优化器 | Adam (lr=0.001) | 自适应学习率，收敛快 |
| 损失函数 | Sparse Categorical Crossentropy | 整数标签多分类标配 |
| 评估指标 | Accuracy | 直接反映分类正确率 |

> **Sparse vs 非 Sparse**：标签是 0-9 整数 → 用 Sparse；若是 one-hot [0,0,1,0,...] → 用 Categorical。

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("模型编译完成: Adam + SparseCategoricalCrossentropy + Accuracy")

---
## 6. 模型训练

In [ ]:
EPOCHS = 10
BATCH_SIZE = 64

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_valid, y_valid),
    verbose=1
)

print(f"\n训练完成! train_acc={history.history['accuracy'][-1]:.4f}  val_acc={history.history['val_accuracy'][-1]:.4f}")

### 6.1 训练曲线

Loss 和 Accuracy 随 epoch 的变化趋势。两条线接近 → 无过拟合；分叉 → 过拟合。

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['loss'], 'o-', label='训练损失', lw=2, ms=4)
ax1.plot(history.history['val_loss'], 's-', label='验证损失', lw=2, ms=4)
ax1.set_title('Loss 曲线', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history.history['accuracy'], 'o-', label='训练准确率', lw=2, ms=4)
ax2.plot(history.history['val_accuracy'], 's-', label='验证准确率', lw=2, ms=4)
ax2.set_title('Accuracy 曲线', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 7. 模型评估

在测试集上评估泛化能力，绘制混淆矩阵和分类报告。

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"{'='*50}")
print(f"测试集 Loss:     {test_loss:.4f}")
print(f"测试集 Accuracy:  {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"{'='*50}")

# 全量预测
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

### 7.1 混淆矩阵

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('混淆矩阵', fontsize=16, fontweight='bold')
plt.xlabel('预测'); plt.ylabel('真实')
plt.tight_layout(); plt.show()

# 各类别准确率
print("各类别识别率:")
for i in range(10):
    tp = cm[i, i]
    total = cm[i, :].sum()
    print(f"  数字 {i}: {tp}/{total} = {tp/total*100:5.1f}%  {'⚠️' if tp/total < 0.97 else '✓'}")

### 7.2 分类报告

In [ ]:
print(classification_report(y_test, y_pred,
    target_names=[str(i) for i in range(10)]))

---
## 8. 预测验证与错误分析

In [ ]:
# 随机 12 张预测验证
fig, axes = plt.subplots(3, 4, figsize=(12, 8))
sample_idx = np.random.choice(len(X_test), 12, replace=False)

for i, ax in enumerate(axes.flat):
    idx = sample_idx[i]
    correct = y_test[idx] == y_pred[idx]
    color = 'green' if correct else 'red'
    ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f'真:{y_test[idx]} 预:{y_pred[idx]}', color=color, fontsize=12, fontweight='bold')
    ax.axis('off')

fig.suptitle('预测验证（绿色=正确，红色=错误）', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

### 8.1 错误案例

In [ ]:
error_idx = np.where(y_pred != y_test)[0]
print(f"错误数: {len(error_idx)} / {len(y_test)} ({len(error_idx)/len(y_test)*100:.2f}%)")

n_show = min(9, len(error_idx))
if n_show > 0:
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    for i, ax in enumerate(axes.flat):
        if i < n_show:
            idx = error_idx[i]
            ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
            ax.set_title(f'真:{y_test[idx]}→预:{y_pred[idx]}', color='red', fontsize=11)
        ax.axis('off')
    fig.suptitle(f'错误案例（共 {len(error_idx)} 个）', fontsize=16, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print("完美！无错误。")

---
## 9. 保存与加载模型

In [ ]:
MODEL_PATH = 'mnist_cnn.h5'
model.save(MODEL_PATH)
print(f"模型已保存: {os.path.abspath(MODEL_PATH)}")
print(f"文件大小: {os.path.getsize(MODEL_PATH)/1024:.1f} KB")

In [ ]:
# 验证加载
loaded = keras.models.load_model(MODEL_PATH)
l_loss, l_acc = loaded.evaluate(X_test, y_test, verbose=0)
print(f"原始模型准确率: {test_acc:.4f}")
print(f"加载模型准确率: {l_acc:.4f}")
print(f"一致性: {'✓' if abs(test_acc - l_acc) < 1e-4 else '✗ 不一致！'}")

In [ ]:
# ⚡ 转换为 ONNX 格式（供 GUI 上位机使用，避免 TensorFlow DLL 冲突）
try:
    import tf2onnx
    spec = (tf.TensorSpec((None, 28, 28, 1), tf.float32, name='input'),)
    model_proto, _ = tf2onnx.convert.from_keras(model, input_signature=spec, opset=13)
    with open('mnist_cnn.onnx', 'wb') as f:
        f.write(model_proto.SerializeToString())
    print(f"ONNX 模型已保存: mnist_cnn.onnx ({os.path.getsize('mnist_cnn.onnx')/1024:.0f} KB)")
except ImportError:
    print("未安装 tf2onnx，跳过 ONNX 转换。GUI 会自动从 .h5 转换。")

---
## 10. 调参实验区

可尝试调整的超参数（建议每次只改一个）：

| 参数 | 默认值 | 方向 |
|------|--------|------|
| 学习率 | 0.001 | 0.0001 ~ 0.01 |
| 卷积核数量 | (32, 64) | (16, 32) / (64, 128) |
| Dropout | 0.5 | 0.3 ~ 0.7 |
| 全连接神经元 | 128 | 64 / 256 |
| Epochs | 10 | 15 ~ 20 |
| Batch Size | 64 | 32 ~ 128 |
| 激活函数 | ReLU | LeakyReLU, ELU |
| 数据增强 | 无 | ImageDataGenerator(rotation, zoom...) |

### 实验模板

In [ ]:
# 💡 复制上面的模型代码到这里，修改参数后运行
#
# 示例：增加卷积核数量 + 数据增强
#
# from tensorflow.keras.preprocessing.image import ImageDataGenerator
# datagen = ImageDataGenerator(
#     rotation_range=10, width_shift_range=0.1,
#     height_shift_range=0.1, zoom_range=0.1)
# datagen.fit(X_train)
#
# model_v2 = keras.Sequential([
#     layers.Conv2D(64, (3,3), activation='relu', input_shape=(28,28,1)),
#     layers.MaxPooling2D((2,2)),
#     layers.Conv2D(128, (3,3), activation='relu'),
#     layers.MaxPooling2D((2,2)),
#     layers.Flatten(),
#     layers.Dense(128, activation='relu'),
#     layers.Dropout(0.5),
#     layers.Dense(10, activation='softmax')
# ])
# model_v2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# history_v2 = model_v2.fit(datagen.flow(X_train, y_train, batch_size=64),
#                           epochs=15, validation_data=(X_valid, y_valid))
# print(f"v2 测试准确率: {model_v2.evaluate(X_test, y_test, verbose=0)[1]:.4f}")

print("调参实验区就绪 — 取消注释上述代码并修改参数运行。")

---

## 总结

✅ 完成 MNIST 手写数字 CNN 分类完整流程：

1. 加载 & 探索数据（70000 条，28×28 灰度，10 类）
2. 预处理（reshape + 合并重分，数据已在 [0,1]）
3. 搭建 CNN（2 卷积 + 2 全连接，~225k 参数）
4. 训练 10 epochs，观察 loss/acc 曲线
5. 测试集评估 + 混淆矩阵 + 分类报告
6. 预测验证 + 错误案例分析
7. 保存模型 (.h5 和 .onnx)
8. 预留调参实验空间

**下一步**: 运行 `python gui_app.py` 启动手写识别上位机！